# 2) Adapter Training (Colab)

Headless continual adapter training with Drive-based artifact output.

In [1]:
from google.colab import drive, userdata

import os
import subprocess
import sys

drive.mount('/content/drive')

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print("[GIT] GitHub token loaded from env/Colab secrets.")
else:
    print("[GIT] No GitHub token found. Auto-push will be skipped unless GH_TOKEN or GITHUB_TOKEN is set.")

if not hf_token:
    print("[HF] No token found. Set a Colab secret or env var named HF_TOKEN before training.")
else:
    try:
        from huggingface_hub import HfApi, login
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
        from huggingface_hub import HfApi, login

    try:
        login(token=hf_token, add_to_git_credential=False)
        profile = dict(HfApi(token=hf_token).whoami() or {})
        username = str(
            profile.get("name")
            or profile.get("fullname")
            or profile.get("email")
            or profile.get("user")
            or "authenticated user"
        )
        print(f"[HF] Authenticated as {username}")
    except Exception as exc:
        print(f"[HF] Authentication check failed: {exc}")
        print("[HF] Training may fail when gated models need authentication.")

Mounted at /content/drive
[GIT] GitHub token loaded from env/Colab secrets.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Authenticated as Grizzmann


In [2]:
import io
import json
import os
import random
import shutil
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: resolve repo root, install deps ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _mount_drive_inline() -> None:
    if not _running_in_colab_inline():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped during bootstrap: {exc}')

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    _mount_drive_inline()
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repository bootstrap failed. Set AADS_REPO_ROOT to an existing checkout, or provide '
        'GH_TOKEN/GITHUB_TOKEN for private GitHub auto-clone.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    mount_drive_if_available,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
NOTEBOOK_FILENAME = '2_interactive_adapter_training.executed.ipynb'
REPO_RUN_DIR = ROOT / 'runs' / RUN_ID
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_training'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_training'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='2_interactive_adapter_training.ipynb',
    run_id=RUN_ID,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR)
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

TELEMETRY.configure_repo_output_export(
    output_dir=REPO_OUTPUT_DIR,
    notebook_filename=NOTEBOOK_FILENAME,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into '/content/bitirmeprojesi'...
Updating files:  20% (185/922)
Updating files:  21% (194/922)
Updating files:  22% (203/922)
Updating files:  23% (213/922)
Updating files:  24% (222/922)
Updating files:  25% (231/922)
Updating files:  26% (240/922)
Updating files:  27% (249/922)
Updating files:  28% (259/922)
Updating files:  29% (268/922)
Updating files:  30% (277/922)
Updating files:  31% (286/922)
Updating files:  32% (296/922)
Updating files:  33% (305/922)
Updating files:  34% (314/922)
Updating files:  35% (323/922)
Updating files:  36% (332/922)
Updating files:  36% (341/922)
Updating files:  37% (342/922)
Updating files:  38% (351/922)
Updating files:  39% (360/922)
Updating files:  40% (369/922)
Updating files:  41% (379/922)
Updating files:  42% (388/922)
Updating files:  43% (397/922)
Updating files:  44% (406/922)
Updating files:  45% (4

In [3]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Training parameters ---
    # Edit these directly, then run the remaining cells top-to-bottom.

    # DATASET_LAYOUT_MODE: 'class_root' prepares and materializes a grouped runtime dataset before training, 'runtime' consumes an already prepared runtime dataset.
    DATASET_LAYOUT_MODE = "runtime"

    # DATASET_ROOT: folder containing class subfolders (one class per subfolder). Used when DATASET_LAYOUT_MODE='class_root' to build the grouped runtime dataset.
    DATASET_ROOT = "data/class_root_dataset"

    # RUNTIME_DATASET_ROOT: runtime dataset root containing <crop>/continual|val|test. Used as the output root in 'class_root' mode and the input root in 'runtime' mode.
    RUNTIME_DATASET_ROOT = "/content/drive/MyDrive/aads_ulora/prepared_runtime_datasets"

    # OOD_ROOT: optional separate folder of unknown examples copied into runtime_dataset/ood/. Ignored when DATASET_LAYOUT_MODE='runtime'.
    OOD_ROOT = ""

    # CROP_NAME: crop key used by taxonomy alignment (e.g., "tomato", "pepper").
    CROP_NAME = "tomato"

    # EPOCHS: total training passes over the train split.
    EPOCHS = 30

    # BATCH_SIZE: samples per optimizer step (increase until near GPU memory limit).
    BATCH_SIZE = 160

    # LEARNING_RATE: optimizer step size for adapter/LoRA parameters.
    LEARNING_RATE = 3e-4

    # LORA_R: LoRA rank (higher = more capacity, more VRAM/compute).
    LORA_R = 16

    # LORA_ALPHA: LoRA scaling factor (commonly 2x to 4x of LORA_R).
    LORA_ALPHA = 32

    # LORA_DROPOUT: dropout applied in LoRA layers (higher = more regularization).
    LORA_DROPOUT = 0.05

    # OOD_FACTOR: multiplier for OOD threshold sensitivity.
    OOD_FACTOR = 2.5
    SURE_SEMANTIC_PERCENTILE = 92.0
    SURE_CONFIDENCE_PERCENTILE = 88.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: experimental training-only regularizer for old/new class energy separation.
    BER_ENABLED = True

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: BER penalty weights for old/new class partitions.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW decoupled weight decay.
    WEIGHT_DECAY = 0.01

    # MIXED_PRECISION: one of {'off', 'auto', 'fp16', 'bf16'}.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation factor.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping threshold (0 disables clipping).
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing coefficient.
    LABEL_SMOOTHING = 0.02

    # Scheduler controls.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.05
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping controls.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.001

    # Reproducibility/runtime toggles.
    DETERMINISTIC = False
    TF32_ENABLED = True

    # NUM_WORKERS: dataloader worker processes (CPU data loading parallelism).
    NUM_WORKERS = 8

    # PREFETCH: dataloader prefetch factor per worker.
    PREFETCH = 4

    # PIN_MEMORY: pin host memory to speed host-to-GPU transfer.
    PIN_MEMORY = True

    # USE_CACHE: keep decoded samples in RAM when supported.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: also cache the continual/train split. Intended for high-RAM Colab runtimes.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: run full validation every N epochs, always including the final epoch.
    VALIDATION_EVERY_N_EPOCHS = 1

    # RESUME_MODE: "fresh" starts a new run, "resume" continues latest checkpoint.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # AUTO_DISCONNECT_RUNTIME: disconnect the Colab runtime after all final exports succeed.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: short delay before disconnect so the final status is visible.
    AUTO_DISCONNECT_GRACE_SECONDS = 60

    # AUTO_PUSH_TO_GITHUB: commit and push runs/<RUN_ID>/ to the repo after final exports complete.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: git remote to push when auto-push is enabled.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: branch override for auto-push. None keeps the current repo branch.
    AUTO_PUSH_BRANCH = None

    # Single profile: high utilization while keeping stability-oriented optimizer settings.
    # If you hit OOM on your dataset, set BATCH_SIZE to 144 in NOTEBOOK_OVERRIDES.
    MAX_STABLE_PROFILE = {
        "EPOCHS": 30,
        "BATCH_SIZE": 160,
        "LEARNING_RATE": 3e-4,
        "LORA_R": 16,
        "LORA_ALPHA": 32,
        "LORA_DROPOUT": 0.05,
        "WEIGHT_DECAY": 0.01,
        "MIXED_PRECISION": "bf16",
        "GRAD_ACCUM_STEPS": 1,
        "MAX_GRAD_NORM": 1.0,
        "LABEL_SMOOTHING": 0.02,
        "SCHEDULER_NAME": "cosine",
        "SCHEDULER_WARMUP_RATIO": 0.05,
        "SCHEDULER_MIN_LR": 1e-6,
        "EARLY_STOPPING_PATIENCE": 5,
        "EARLY_STOPPING_MIN_DELTA": 0.001,
        "OOD_FACTOR": 2.5,
        "SURE_SEMANTIC_PERCENTILE": 92.0,
        "SURE_CONFIDENCE_PERCENTILE": 88.0,
        "CONFORMAL_ALPHA": 0.05,
        "CONFORMAL_METHOD": "raps",
        "CONFORMAL_RAPS_LAMBDA": 0.2,
        "CONFORMAL_RAPS_K_REG": 1,
        "NUM_WORKERS": 12,
        "PREFETCH": 8,
        "PIN_MEMORY": True,
        "USE_CACHE": True,
        "CACHE_TRAIN_SPLIT": True,
        "VALIDATION_EVERY_N_EPOCHS": 1,
        "CHECKPOINT_EVERY_N_STEPS": 800,
        "CHECKPOINT_ON_EXCEPTION": True,
    }

    BASE_CONFIG = ConfigurationManager(config_dir=str(ROOT / "config"), environment="colab").load_all_configs()
    COLAB_TRAINING_CFG = BASE_CONFIG.get("colab", {}).get("training", {})
    CONTINUAL_DATA_CFG = BASE_CONFIG.get("training", {}).get("continual", {}).get("data", {})
    SEED = int(BASE_CONFIG.get("training", {}).get("continual", {}).get("seed", 42))
    STDOUT_BATCH_INTERVAL = int(COLAB_TRAINING_CFG.get("stdout_progress_batch_interval", 50))
    CHECKPOINT_EVERY_N_STEPS = int(
        COLAB_TRAINING_CFG.get(
            "checkpoint_every_n_steps",
            COLAB_TRAINING_CFG.get("checkpoint_interval", 200),
        )
    )
    CHECKPOINT_ON_EXCEPTION = bool(COLAB_TRAINING_CFG.get("checkpoint_on_exception", True))

    profile_payload = dict(MAX_STABLE_PROFILE)
    EPOCHS = int(profile_payload.get("EPOCHS", EPOCHS))
    BATCH_SIZE = int(profile_payload.get("BATCH_SIZE", BATCH_SIZE))
    LEARNING_RATE = float(profile_payload.get("LEARNING_RATE", LEARNING_RATE))
    LORA_R = int(profile_payload.get("LORA_R", LORA_R))
    LORA_ALPHA = int(profile_payload.get("LORA_ALPHA", LORA_ALPHA))
    LORA_DROPOUT = float(profile_payload.get("LORA_DROPOUT", LORA_DROPOUT))
    WEIGHT_DECAY = float(profile_payload.get("WEIGHT_DECAY", WEIGHT_DECAY))
    MIXED_PRECISION = str(profile_payload.get("MIXED_PRECISION", MIXED_PRECISION))
    GRAD_ACCUM_STEPS = int(profile_payload.get("GRAD_ACCUM_STEPS", GRAD_ACCUM_STEPS))
    MAX_GRAD_NORM = float(profile_payload.get("MAX_GRAD_NORM", MAX_GRAD_NORM))
    LABEL_SMOOTHING = float(profile_payload.get("LABEL_SMOOTHING", LABEL_SMOOTHING))
    SCHEDULER_NAME = str(profile_payload.get("SCHEDULER_NAME", SCHEDULER_NAME))
    SCHEDULER_WARMUP_RATIO = float(profile_payload.get("SCHEDULER_WARMUP_RATIO", SCHEDULER_WARMUP_RATIO))
    SCHEDULER_MIN_LR = float(profile_payload.get("SCHEDULER_MIN_LR", SCHEDULER_MIN_LR))
    EARLY_STOPPING_PATIENCE = int(profile_payload.get("EARLY_STOPPING_PATIENCE", EARLY_STOPPING_PATIENCE))
    EARLY_STOPPING_MIN_DELTA = float(profile_payload.get("EARLY_STOPPING_MIN_DELTA", EARLY_STOPPING_MIN_DELTA))
    OOD_FACTOR = float(profile_payload.get("OOD_FACTOR", OOD_FACTOR))
    SURE_SEMANTIC_PERCENTILE = float(profile_payload.get("SURE_SEMANTIC_PERCENTILE", SURE_SEMANTIC_PERCENTILE))
    SURE_CONFIDENCE_PERCENTILE = float(profile_payload.get("SURE_CONFIDENCE_PERCENTILE", SURE_CONFIDENCE_PERCENTILE))
    CONFORMAL_ALPHA = float(profile_payload.get("CONFORMAL_ALPHA", CONFORMAL_ALPHA))
    CONFORMAL_METHOD = str(profile_payload.get("CONFORMAL_METHOD", CONFORMAL_METHOD))
    CONFORMAL_RAPS_LAMBDA = float(profile_payload.get("CONFORMAL_RAPS_LAMBDA", CONFORMAL_RAPS_LAMBDA))
    CONFORMAL_RAPS_K_REG = int(profile_payload.get("CONFORMAL_RAPS_K_REG", CONFORMAL_RAPS_K_REG))
    NUM_WORKERS = int(profile_payload.get("NUM_WORKERS", NUM_WORKERS))
    PREFETCH = int(profile_payload.get("PREFETCH", PREFETCH))
    PIN_MEMORY = bool(profile_payload.get("PIN_MEMORY", PIN_MEMORY))
    USE_CACHE = bool(profile_payload.get("USE_CACHE", USE_CACHE))
    CACHE_TRAIN_SPLIT = bool(profile_payload.get("CACHE_TRAIN_SPLIT", CACHE_TRAIN_SPLIT))
    VALIDATION_EVERY_N_EPOCHS = int(profile_payload.get("VALIDATION_EVERY_N_EPOCHS", VALIDATION_EVERY_N_EPOCHS))
    CHECKPOINT_EVERY_N_STEPS = int(profile_payload.get("CHECKPOINT_EVERY_N_STEPS", CHECKPOINT_EVERY_N_STEPS))
    CHECKPOINT_ON_EXCEPTION = bool(profile_payload.get("CHECKPOINT_ON_EXCEPTION", CHECKPOINT_ON_EXCEPTION))
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = bool(TF32_ENABLED)

    # Optional final override map (highest precedence). Example:
    # NOTEBOOK_OVERRIDES = {"BATCH_SIZE": 48, "NUM_WORKERS": 10}
    NOTEBOOK_OVERRIDES = {}
    if NOTEBOOK_OVERRIDES:
        EPOCHS = int(NOTEBOOK_OVERRIDES.get("EPOCHS", EPOCHS))
        BATCH_SIZE = int(NOTEBOOK_OVERRIDES.get("BATCH_SIZE", BATCH_SIZE))
        LEARNING_RATE = float(NOTEBOOK_OVERRIDES.get("LEARNING_RATE", LEARNING_RATE))
        LORA_R = int(NOTEBOOK_OVERRIDES.get("LORA_R", LORA_R))
        LORA_ALPHA = int(NOTEBOOK_OVERRIDES.get("LORA_ALPHA", LORA_ALPHA))
        LORA_DROPOUT = float(NOTEBOOK_OVERRIDES.get("LORA_DROPOUT", LORA_DROPOUT))
        WEIGHT_DECAY = float(NOTEBOOK_OVERRIDES.get("WEIGHT_DECAY", WEIGHT_DECAY))
        MIXED_PRECISION = str(NOTEBOOK_OVERRIDES.get("MIXED_PRECISION", MIXED_PRECISION))
        GRAD_ACCUM_STEPS = int(NOTEBOOK_OVERRIDES.get("GRAD_ACCUM_STEPS", GRAD_ACCUM_STEPS))
        MAX_GRAD_NORM = float(NOTEBOOK_OVERRIDES.get("MAX_GRAD_NORM", MAX_GRAD_NORM))
        LABEL_SMOOTHING = float(NOTEBOOK_OVERRIDES.get("LABEL_SMOOTHING", LABEL_SMOOTHING))
        SCHEDULER_NAME = str(NOTEBOOK_OVERRIDES.get("SCHEDULER_NAME", SCHEDULER_NAME))
        SCHEDULER_WARMUP_RATIO = float(NOTEBOOK_OVERRIDES.get("SCHEDULER_WARMUP_RATIO", SCHEDULER_WARMUP_RATIO))
        SCHEDULER_MIN_LR = float(NOTEBOOK_OVERRIDES.get("SCHEDULER_MIN_LR", SCHEDULER_MIN_LR))
        EARLY_STOPPING_PATIENCE = int(NOTEBOOK_OVERRIDES.get("EARLY_STOPPING_PATIENCE", EARLY_STOPPING_PATIENCE))
        EARLY_STOPPING_MIN_DELTA = float(NOTEBOOK_OVERRIDES.get("EARLY_STOPPING_MIN_DELTA", EARLY_STOPPING_MIN_DELTA))
        OOD_FACTOR = float(NOTEBOOK_OVERRIDES.get("OOD_FACTOR", OOD_FACTOR))
        SURE_SEMANTIC_PERCENTILE = float(NOTEBOOK_OVERRIDES.get("SURE_SEMANTIC_PERCENTILE", SURE_SEMANTIC_PERCENTILE))
        SURE_CONFIDENCE_PERCENTILE = float(NOTEBOOK_OVERRIDES.get("SURE_CONFIDENCE_PERCENTILE", SURE_CONFIDENCE_PERCENTILE))
        CONFORMAL_ALPHA = float(NOTEBOOK_OVERRIDES.get("CONFORMAL_ALPHA", CONFORMAL_ALPHA))
        CONFORMAL_METHOD = str(NOTEBOOK_OVERRIDES.get("CONFORMAL_METHOD", CONFORMAL_METHOD))
        CONFORMAL_RAPS_LAMBDA = float(NOTEBOOK_OVERRIDES.get("CONFORMAL_RAPS_LAMBDA", CONFORMAL_RAPS_LAMBDA))
        CONFORMAL_RAPS_K_REG = int(NOTEBOOK_OVERRIDES.get("CONFORMAL_RAPS_K_REG", CONFORMAL_RAPS_K_REG))
        NUM_WORKERS = int(NOTEBOOK_OVERRIDES.get("NUM_WORKERS", NUM_WORKERS))
        PREFETCH = int(NOTEBOOK_OVERRIDES.get("PREFETCH", PREFETCH))
        PIN_MEMORY = bool(NOTEBOOK_OVERRIDES.get("PIN_MEMORY", PIN_MEMORY))
        USE_CACHE = bool(NOTEBOOK_OVERRIDES.get("USE_CACHE", USE_CACHE))
        CACHE_TRAIN_SPLIT = bool(NOTEBOOK_OVERRIDES.get("CACHE_TRAIN_SPLIT", CACHE_TRAIN_SPLIT))
        VALIDATION_EVERY_N_EPOCHS = int(NOTEBOOK_OVERRIDES.get("VALIDATION_EVERY_N_EPOCHS", VALIDATION_EVERY_N_EPOCHS))
        CHECKPOINT_EVERY_N_STEPS = int(NOTEBOOK_OVERRIDES.get("CHECKPOINT_EVERY_N_STEPS", CHECKPOINT_EVERY_N_STEPS))
        CHECKPOINT_ON_EXCEPTION = bool(NOTEBOOK_OVERRIDES.get("CHECKPOINT_ON_EXCEPTION", CHECKPOINT_ON_EXCEPTION))
    TARGET_SIZE = int(CONTINUAL_DATA_CFG.get("target_size", 224))
    DATA_SAMPLER = str(CONTINUAL_DATA_CFG.get("sampler", "shuffle"))
    LOADER_ERROR_POLICY = str(CONTINUAL_DATA_CFG.get("loader_error_policy", "tolerant"))
    CACHE_SIZE = int(CONTINUAL_DATA_CFG.get("cache_size", 1000))
    CACHE_TRAIN_SPLIT = bool(CONTINUAL_DATA_CFG.get("cache_train_split", CACHE_TRAIN_SPLIT))
    VALIDATE_IMAGES_ON_INIT = bool(CONTINUAL_DATA_CFG.get("validate_images_on_init", True))

    STATE = {
        "validated": False,
        "class_names": [],
        "runtime_dataset_root": None,
        "adapter": None,
        "loaders": None,
        "history": None,
        "calibration": None,
        "checkpoint_manager": CHECKPOINT_MANAGER,
        "resume_manifest": None,
        "best_val_loss": None,
        "best_metric_state": {},
        "auto_disconnect_report": None,
        "git_push_report": None,
        "telemetry": TELEMETRY,
    }

    if RESUME_MODE == "resume":
        try:
            STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
            if STATE["resume_manifest"]:
                manifest = STATE["resume_manifest"]
                print(
                    f"[RESUME] Found checkpoint: {manifest.get('name', '?')} "
                    f"epoch={manifest.get('epoch', 0)} step={manifest.get('global_step', 0)}"
                )
        except Exception:
            pass

    hf_token_ready = False
    hf_token = str(resolve_hf_token() or "").strip()
    if not hf_token:
        raise RuntimeError(
            "Notebook 2 requires a Hugging Face token before training starts. "
            "Set HF_TOKEN, HUGGINGFACE_TOKEN, or HUGGINGFACE_HUB_TOKEN in env vars or Colab secrets."
        )
    hf_token_ready = bool(login_and_check_hf_token(print_fn=print))
    if not hf_token_ready:
        raise RuntimeError(
            "Hugging Face token preflight failed. Fix the token before starting training."
        )

    github_token_ready = False
    if AUTO_PUSH_TO_GITHUB:
        github_token = str(resolve_github_token() or "").strip()
        if not github_token:
            raise RuntimeError(
                "AUTO_PUSH_TO_GITHUB is enabled, but no GitHub token was found. "
                "Set GH_TOKEN or GITHUB_TOKEN in env vars or Colab secrets before training starts."
            )
        github_token_ready = True
        print("[GIT] Auto-push preflight passed: GitHub token resolved.")

    print(
        f"[CONFIG] crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} "
        f"lr={LEARNING_RATE} resume={RESUME_MODE} ber={BER_ENABLED} "
        f"auto_disconnect={AUTO_DISCONNECT_RUNTIME} auto_push={AUTO_PUSH_TO_GITHUB}"
    )
    print(
        f"[RUNTIME] profile=max_stable mp={MIXED_PRECISION} workers={NUM_WORKERS} prefetch={PREFETCH} "
        f"sched={SCHEDULER_NAME} wd={WEIGHT_DECAY} accum={GRAD_ACCUM_STEPS} grad_clip={MAX_GRAD_NORM} "
        f"label_smooth={LABEL_SMOOTHING} warmup={SCHEDULER_WARMUP_RATIO} "
        f"early_stop={EARLY_STOPPING_PATIENCE}/{EARLY_STOPPING_MIN_DELTA} "
        f"val_every={VALIDATION_EVERY_N_EPOCHS} cache_train={CACHE_TRAIN_SPLIT}"
    )
    print(
        f"[OOD] factor={OOD_FACTOR} sure={SURE_SEMANTIC_PERCENTILE}/{SURE_CONFIDENCE_PERCENTILE} "
        f"conformal={CONFORMAL_METHOD} alpha={CONFORMAL_ALPHA} "
        f"raps_lambda={CONFORMAL_RAPS_LAMBDA} raps_k={CONFORMAL_RAPS_K_REG}"
    )
    TELEMETRY.update_latest(
        {
            "phase": "parameters_ready",
            "crop": CROP_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "mixed_precision": MIXED_PRECISION,
            "num_workers": NUM_WORKERS,
            "prefetch": PREFETCH,
            "cache_train_split": CACHE_TRAIN_SPLIT,
            "ber_enabled": BER_ENABLED,
            "ber_lambda_old": BER_LAMBDA_OLD,
            "ber_lambda_new": BER_LAMBDA_NEW,
            "ber_warmup_steps": BER_WARMUP_STEPS,
            "ood_factor": OOD_FACTOR,
            "sure_semantic_percentile": SURE_SEMANTIC_PERCENTILE,
            "sure_confidence_percentile": SURE_CONFIDENCE_PERCENTILE,
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_method": CONFORMAL_METHOD,
            "conformal_raps_lambda": CONFORMAL_RAPS_LAMBDA,
            "conformal_raps_k_reg": CONFORMAL_RAPS_K_REG,
            "label_smoothing": LABEL_SMOOTHING,
            "scheduler_warmup_ratio": SCHEDULER_WARMUP_RATIO,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "resume_mode": RESUME_MODE,
            "validation_every_n_epochs": VALIDATION_EVERY_N_EPOCHS,
            "auto_push_to_github": AUTO_PUSH_TO_GITHUB,
            "hf_token_ready": hf_token_ready,
            "auto_push_token_ready": github_token_ready,
            "auto_push_remote_name": AUTO_PUSH_REMOTE_NAME,
            "auto_push_branch": AUTO_PUSH_BRANCH,
            "auto_disconnect_runtime": AUTO_DISCONNECT_RUNTIME,
            "auto_disconnect_grace_seconds": AUTO_DISCONNECT_GRACE_SECONDS,
        }
    )

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Authenticated as Grizzmann
[GIT] Auto-push preflight passed: GitHub token resolved.
[CONFIG] crop=tomato epochs=30 bs=160 lr=0.0003 resume=fresh ber=True auto_disconnect=True auto_push=True
[RUNTIME] profile=max_stable mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.01 accum=1 grad_clip=1.0 label_smooth=0.02 warmup=0.05 early_stop=5/0.001 val_every=1 cache_train=True
[OOD] factor=2.5 sure=92.0/88.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [4]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Validation"):
    layout_mode = str(DATASET_LAYOUT_MODE).strip().lower()
    if layout_mode not in {"class_root", "runtime"}:
        raise RuntimeError(f"Unsupported DATASET_LAYOUT_MODE: {DATASET_LAYOUT_MODE}")

    crop_key = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(CROP_NAME).strip())
    while "__" in crop_key:
        crop_key = crop_key.replace("__", "_")
    crop_key = crop_key.strip("_")
    if not crop_key:
        raise RuntimeError("CROP_NAME must resolve to a non-empty crop key.")

    if layout_mode == "runtime":
        runtime_root = Path(RUNTIME_DATASET_ROOT).expanduser()
        if not runtime_root.is_absolute():
            runtime_root = (ROOT / runtime_root).resolve()
        crop_root = runtime_root / crop_key
        if not crop_root.is_dir():
            raise RuntimeError(f"Prepared runtime crop root not found: {crop_root}")
        missing_splits = [name for name in ("continual", "val", "test") if not (crop_root / name).is_dir()]
        if missing_splits:
            raise RuntimeError(f"Prepared runtime dataset is missing split folder(s): {missing_splits}")
        class_names = sorted(d.name for d in (crop_root / "continual").iterdir() if d.is_dir())
        if not class_names:
            raise RuntimeError(f"No class subdirectories in prepared runtime split: {crop_root / 'continual'}")
        STATE["class_names"] = class_names
        STATE["validated"] = True
        STATE["dataset_layout_mode"] = layout_mode
        print(f"[DATASET] mode=runtime  root={crop_root}  classes={len(class_names)}: {class_names}")
        TELEMETRY.update_latest(
            {
                "phase": "dataset_validated",
                "dataset_layout_mode": layout_mode,
                "runtime_dataset_root": str(runtime_root),
                "class_count": len(class_names),
            }
        )
    else:
        raw_root = Path(DATASET_ROOT).expanduser()
        if not raw_root.is_absolute():
            raw_root = (ROOT / raw_root).resolve()
        if not raw_root.is_dir():
            raise RuntimeError(f"Dataset root not found: {raw_root}")

        class_names = sorted(d.name for d in raw_root.iterdir() if d.is_dir())
        if not class_names:
            raise RuntimeError(f"No class subdirectories in {raw_root}")

        STATE["class_names"] = class_names
        STATE["validated"] = True
        STATE["dataset_layout_mode"] = layout_mode
        print(f"[DATASET] mode=class_root  root={raw_root}  classes={len(class_names)}: {class_names}")
        TELEMETRY.update_latest(
            {
                "phase": "dataset_validated",
                "dataset_layout_mode": layout_mode,
                "dataset_root": str(raw_root),
                "class_count": len(class_names),
            }
        )


[DATASET] mode=runtime  root=/content/drive/MyDrive/aads_ulora/prepared_runtime_datasets/tomato  classes=12: ['healthy', 'tomato_bacterial_spot_and_speck_leaf', 'tomato_early_blight_leaf', 'tomato_gray_mold_leaf', 'tomato_late_blight_leaf', 'tomato_leaf_mold_leaf', 'tomato_mosaic_virus_leaf', 'tomato_powdery_mildew_leaf', 'tomato_septoria_leaf_spot_leaf', 'tomato_spider_mites_leaf', 'tomato_spotted_wilt_virus', 'tomato_yellow_leaf_curl_leaf']


In [5]:
with TELEMETRY.capture_cell_output("Cell 5: Engine Init"):
    from scripts.colab_dataset_layout import resolve_notebook_training_classes
    from scripts.prepare_grouped_runtime_dataset import (
        DEFAULT_RUNTIME_ROOT,
        build_grouped_dataset_plan,
        materialize_grouped_runtime_dataset,
    )
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.utils.data_loader import create_training_loaders

    def _normalize(name: str) -> str:
        normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in name.strip())
        while "__" in normalized:
            normalized = normalized.replace("__", "_")
        return normalized.strip("_")

    def _copy_ood_into_runtime(*, ood_root: Path, runtime_data_root: Path, crop_name: str) -> None:
        import shutil

        destination_root = runtime_data_root / crop_name / "ood"
        if destination_root.exists():
            shutil.rmtree(destination_root)
        destination_root.mkdir(parents=True, exist_ok=True)
        image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
        copied = 0
        for source_path in sorted(path for path in ood_root.rglob("*") if path.is_file() and path.suffix.lower() in image_exts):
            destination_path = destination_root / source_path.relative_to(ood_root)
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)
            copied += 1
        print(f"[DATASET] Copied OOD examples into runtime dataset: {copied}")

    if not STATE.get("validated"):
        raise RuntimeError("Run dataset validation cell first.")

    crop_name = CROP_NAME.strip().lower()
    layout_mode = str(STATE.get("dataset_layout_mode", DATASET_LAYOUT_MODE)).strip().lower()
    ood_root = None
    if layout_mode == "runtime":
        runtime_data_root = Path(RUNTIME_DATASET_ROOT).expanduser()
        if not runtime_data_root.is_absolute():
            runtime_data_root = (ROOT / runtime_data_root).resolve()
        class_root = runtime_data_root / crop_name / "continual"
        if not class_root.is_dir():
            raise RuntimeError(f"Prepared runtime continual split not found: {class_root}")
        available = sorted({_normalize(path.name) for path in class_root.iterdir() if path.is_dir() and _normalize(path.name)})
        if str(OOD_ROOT).strip():
            print("[DATASET] OOD_ROOT is ignored when DATASET_LAYOUT_MODE='runtime'.")
    else:
        class_root = Path(DATASET_ROOT).expanduser()
        if not class_root.is_absolute():
            class_root = (ROOT / class_root).resolve()
        available = sorted({_normalize(path.name) for path in class_root.iterdir() if path.is_dir() and _normalize(path.name)})
        if str(OOD_ROOT).strip():
            ood_root = Path(OOD_ROOT).expanduser()
            if not ood_root.is_absolute():
                ood_root = (ROOT / ood_root).resolve()
            if not ood_root.exists():
                raise RuntimeError(f"OOD root not found: {ood_root}")

    class_resolution = resolve_notebook_training_classes(
        available_classes=available,
        crop_name=crop_name,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
    )
    aligned = list(class_resolution.get("selected_classes", available))

    if not aligned:
        raise RuntimeError(f"No usable classes for crop '{crop_name}'. Available: {available}")

    if layout_mode == "runtime":
        final_class_names = aligned
        STATE["runtime_dataset_root"] = runtime_data_root
    else:
        prep_artifact_root = ROOT / "outputs" / "colab_notebook_data_prep" / "artifacts"
        runtime_data_root = Path(RUNTIME_DATASET_ROOT or DEFAULT_RUNTIME_ROOT).expanduser()
        if not runtime_data_root.is_absolute():
            runtime_data_root = (ROOT / runtime_data_root).resolve()

        print("[DATASET] Building grouped prep plan before training.")
        prep_summary = build_grouped_dataset_plan(
            class_root=class_root,
            crop_name=crop_name,
            artifact_root=prep_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            device=DEVICE,
            batch_size=max(16, int(BATCH_SIZE)),
            neighbors=4,
            progress_fn=print,
        )
        print(f"[DATASET] grouped_prep runtime_ready={prep_summary.get('runtime_ready')} cross_class_conflicts={prep_summary.get('summary', {}).get('cross_class_conflicts')} same_class_high_risk_clusters={prep_summary.get('summary', {}).get('same_class_high_risk_clusters')}")
        if not prep_summary.get("runtime_ready"):
            raise RuntimeError(
                "Notebook 2 grouped data preparation found blockers. Review outputs/colab_notebook_data_prep/artifacts or run Notebook 0 for audit-first cleanup."
            )

        materialize_grouped_runtime_dataset(
            class_root=class_root,
            crop_name=crop_name,
            artifact_root=prep_artifact_root,
            runtime_root=runtime_data_root,
            materialization_strategy="auto",
        )
        if ood_root is not None:
            _copy_ood_into_runtime(ood_root=ood_root, runtime_data_root=runtime_data_root, crop_name=crop_name)
        class_root = runtime_data_root / crop_name / "continual"
        if not class_root.is_dir():
            raise RuntimeError(f"Prepared runtime continual split not found after materialization: {class_root}")
        final_class_names = sorted({_normalize(path.name) for path in class_root.iterdir() if path.is_dir() and _normalize(path.name)})
        STATE["grouped_prep_summary"] = prep_summary
        STATE["runtime_dataset_root"] = runtime_data_root
        print(f"[DATASET] Grouped runtime dataset materialized under {runtime_data_root / crop_name}")

    STATE["class_names"] = final_class_names
    STATE["class_resolution"] = class_resolution
    print(f"[CLASSES] {final_class_names}")
    print(
        f"[CLASSES] mode={'taxonomy_filter' if class_resolution.get('used_taxonomy_filter') else 'dataset_fallback'} "
        f"reason={class_resolution.get('reason', 'unknown')} "
        f"matched={len(class_resolution.get('matched_classes', []))} "
        f"unmatched={len(class_resolution.get('unmatched_classes', []))}"
    )
    if class_resolution.get("unmatched_classes"):
        print(f"[CLASSES] taxonomy-unmatched classes kept: {class_resolution['unmatched_classes']}")

    continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
    continual_cfg["device"] = DEVICE
    continual_cfg["num_epochs"] = EPOCHS
    continual_cfg["batch_size"] = BATCH_SIZE
    continual_cfg["learning_rate"] = LEARNING_RATE
    continual_cfg["adapter"]["lora_r"] = LORA_R
    continual_cfg["adapter"]["lora_alpha"] = LORA_ALPHA
    continual_cfg["adapter"]["lora_dropout"] = LORA_DROPOUT
    continual_cfg["weight_decay"] = WEIGHT_DECAY
    continual_cfg["deterministic"] = bool(DETERMINISTIC)
    continual_cfg["ood"]["threshold_factor"] = OOD_FACTOR
    continual_cfg["ood"]["sure_semantic_percentile"] = SURE_SEMANTIC_PERCENTILE
    continual_cfg["ood"]["sure_confidence_percentile"] = SURE_CONFIDENCE_PERCENTILE
    continual_cfg["ood"]["conformal_alpha"] = CONFORMAL_ALPHA
    continual_cfg["ood"]["conformal_method"] = CONFORMAL_METHOD
    continual_cfg["ood"]["conformal_raps_lambda"] = CONFORMAL_RAPS_LAMBDA
    continual_cfg["ood"]["conformal_raps_k_reg"] = CONFORMAL_RAPS_K_REG
    continual_cfg["ood"]["ber_enabled"] = BER_ENABLED
    continual_cfg["ood"]["ber_lambda_old"] = BER_LAMBDA_OLD
    continual_cfg["ood"]["ber_lambda_new"] = BER_LAMBDA_NEW
    continual_cfg["ood"]["ber_warmup_steps"] = int(BER_WARMUP_STEPS)
    print(f"[ENGINE][OOD_CFG] {json.dumps(continual_cfg['ood'], sort_keys=True)}")

    optimization_cfg = continual_cfg.setdefault("optimization", {})
    optimization_cfg["grad_accumulation_steps"] = int(GRAD_ACCUM_STEPS)
    optimization_cfg["max_grad_norm"] = float(MAX_GRAD_NORM)
    optimization_cfg["mixed_precision"] = str(MIXED_PRECISION)
    optimization_cfg["label_smoothing"] = float(LABEL_SMOOTHING)
    scheduler_cfg = optimization_cfg.setdefault("scheduler", {})
    scheduler_cfg["name"] = str(SCHEDULER_NAME)
    scheduler_cfg["warmup_ratio"] = float(SCHEDULER_WARMUP_RATIO)
    scheduler_cfg["min_lr"] = float(SCHEDULER_MIN_LR)
    scheduler_cfg["step_on"] = str(scheduler_cfg.get("step_on", "batch"))
    early_stopping_cfg = continual_cfg.setdefault("early_stopping", {})
    early_stopping_cfg["enabled"] = True
    early_stopping_cfg["patience"] = int(EARLY_STOPPING_PATIENCE)
    early_stopping_cfg["min_delta"] = float(EARLY_STOPPING_MIN_DELTA)

    adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
    print("[ENGINE] Initializing adapter (may download backbone)...")
    adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})

    loader_kwargs = {}
    if NUM_WORKERS > 0:
        loader_kwargs["prefetch_factor"] = PREFETCH

    loaders = create_training_loaders(
        data_dir=str(runtime_data_root),
        crop=crop_name,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        use_cache=USE_CACHE,
        cache_size=CACHE_SIZE,
        cache_train_split=CACHE_TRAIN_SPLIT,
        target_size=TARGET_SIZE,
        error_policy=LOADER_ERROR_POLICY,
        sampler=DATA_SAMPLER,
        seed=SEED,
        validate_images_on_init=VALIDATE_IMAGES_ON_INIT,
        pin_memory=PIN_MEMORY,
        **loader_kwargs,
    )

    STATE["adapter"] = adapter
    STATE["loaders"] = loaders
    STATE["continual_config"] = continual_cfg

    trainable = sum(parameter.numel() for parameter in adapter.parameters() if parameter.requires_grad)
    print(f"[ENGINE] Ready. trainable_params={trainable:,}  classes={len(final_class_names)}")
    TELEMETRY.update_latest(
        {
            "phase": "engine_ready",
            "class_count": len(final_class_names),
            "runtime_dataset_root": str(runtime_data_root),
            "dataset_layout_mode": layout_mode,
        }
    )

[CLASSES] ['healthy', 'tomato_bacterial_spot_and_speck_leaf', 'tomato_early_blight_leaf', 'tomato_gray_mold_leaf', 'tomato_late_blight_leaf', 'tomato_leaf_mold_leaf', 'tomato_mosaic_virus_leaf', 'tomato_powdery_mildew_leaf', 'tomato_septoria_leaf_spot_leaf', 'tomato_spider_mites_leaf', 'tomato_spotted_wilt_virus', 'tomato_yellow_leaf_curl_leaf']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=2 unmatched=10
[CLASSES] taxonomy-unmatched classes kept: ['tomato_bacterial_spot_and_speck_leaf', 'tomato_early_blight_leaf', 'tomato_gray_mold_leaf', 'tomato_late_blight_leaf', 'tomato_leaf_mold_leaf', 'tomato_mosaic_virus_leaf', 'tomato_powdery_mildew_leaf', 'tomato_septoria_leaf_spot_leaf', 'tomato_spider_mites_leaf', 'tomato_yellow_leaf_curl_leaf']
[ENGINE][OOD_CFG] {"ber_enabled": true, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg":

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Ready. trainable_params=10,237,456  classes=12


In [6]:
with TELEMETRY.capture_cell_output("Cell 5b: OOD Config Verify"):
    if "continual_config" not in STATE:
        raise RuntimeError("Run Cell 5 first so continual_config is available.")

    resolved_ood_cfg = dict(STATE["continual_config"].get("ood", {}))
    expected_ood_cfg = {
        "threshold_factor": float(OOD_FACTOR),
        "sure_semantic_percentile": float(SURE_SEMANTIC_PERCENTILE),
        "sure_confidence_percentile": float(SURE_CONFIDENCE_PERCENTILE),
        "conformal_alpha": float(CONFORMAL_ALPHA),
        "conformal_method": str(CONFORMAL_METHOD),
        "conformal_raps_lambda": float(CONFORMAL_RAPS_LAMBDA),
        "conformal_raps_k_reg": int(CONFORMAL_RAPS_K_REG),
    }

    print("[VERIFY][OOD][expected]", json.dumps(expected_ood_cfg, sort_keys=True))
    print("[VERIFY][OOD][resolved]", json.dumps({k: resolved_ood_cfg.get(k) for k in expected_ood_cfg}, sort_keys=True))

    mismatches = []
    for key, expected in expected_ood_cfg.items():
        actual = resolved_ood_cfg.get(key)
        if isinstance(expected, float):
            try:
                actual_float = float(actual)
            except Exception:
                mismatches.append(f"{key}: expected={expected} actual={actual}")
                continue
            if abs(actual_float - expected) > 1e-12:
                mismatches.append(f"{key}: expected={expected} actual={actual_float}")
        elif actual != expected:
            mismatches.append(f"{key}: expected={expected} actual={actual}")

    if mismatches:
        raise RuntimeError("OOD config mismatch:\n - " + "\n - ".join(mismatches))

    print("[VERIFY][OOD] OK: resolved OOD config matches requested parameters.")

[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 88.0, "sure_semantic_percentile": 92.0, "threshold_factor": 2.5}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 88.0, "sure_semantic_percentile": 92.0, "threshold_factor": 2.5}
[VERIFY][OOD] OK: resolved OOD config matches requested parameters.


In [7]:
with TELEMETRY.capture_cell_output("Cell 6: Training"):
    from scripts.colab_notebook_helpers import (
        build_history_snapshot,
        persist_training_curve_figure,
        persist_training_history_artifacts,
        save_notebook_checkpoint,
    )

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    checkpoint_manager = STATE["checkpoint_manager"]
    val_loader = loaders.get("val")
    run_id = RUN_ID
    telemetry = STATE.get("telemetry") or TELEMETRY

    resume_state = None
    if RESUME_MODE == "resume" and isinstance(STATE.get("resume_manifest"), dict):
        checkpoint_path = str(STATE["resume_manifest"].get("path", "")).strip()
        if checkpoint_path:
            try:
                resume_state = adapter.load_training_checkpoint(checkpoint_path)
                STATE["resume_state"] = resume_state
                progress_state = resume_state.get("progress_state") or {}
                print(
                    f"[RESUME] epoch={progress_state.get('epoch', 0)} "
                    f"step={progress_state.get('global_step', 0)}"
                )
            except Exception as exc:
                print(f"[RESUME] Failed: {exc}")

    existing_history = (resume_state or {}).get("history", (resume_state or {}).get("history_snapshot", {}))
    train_loss_curve = list(existing_history.get("train_loss", []))
    val_loss_curve = list(existing_history.get("val_loss", []))
    val_acc_curve = list(existing_history.get("val_accuracy", []))
    macro_f1_curve = list(existing_history.get("macro_f1", []))
    weighted_f1_curve = list(existing_history.get("weighted_f1", []))
    balanced_acc_curve = list(existing_history.get("balanced_accuracy", []))
    gap_curve = list(existing_history.get("generalization_gap", []))

    start_time = time.time()
    session = None
    last_checkpoint_step = -1
    best_val_loss = float(STATE["best_val_loss"]) if STATE.get("best_val_loss") is not None else None

    print(f"[TRAIN] epochs={EPOCHS} device={DEVICE} batch_interval={STDOUT_BATCH_INTERVAL}")
    telemetry.update_latest(
        {
            "phase": "training_started",
            "epochs": EPOCHS,
            "batch_interval": STDOUT_BATCH_INTERVAL,
        }
    )

    def _history_snapshot():
        return build_history_snapshot(
            state_history=STATE.get("history"),
            train_loss_curve=train_loss_curve,
            val_loss_curve=val_loss_curve,
            val_acc_curve=val_acc_curve,
            macro_f1_curve=macro_f1_curve,
            weighted_f1_curve=weighted_f1_curve,
            balanced_acc_curve=balanced_acc_curve,
            gap_curve=gap_curve,
        )

    def _persist_history():
        snapshot = _history_snapshot()
        STATE["history"] = dict((STATE.get("history") or {}), **snapshot)
        persist_training_history_artifacts(
            root=ROOT,
            history_snapshot=STATE["history"],
            telemetry=telemetry,
        )
        return snapshot

    def _checkpoint(reason, event, mark_best=False, val_loss=None):
        if session is None:
            return None
        record = save_notebook_checkpoint(
            checkpoint_manager=checkpoint_manager,
            adapter=adapter,
            session=session,
            reason=reason,
            run_id=run_id,
            telemetry=telemetry,
            mark_best=bool(mark_best),
            val_loss=(float(val_loss) if val_loss is not None else None),
        )
        if record is not None:
            STATE["resume_manifest"] = record
            print(f"[CKPT] {reason} epoch={record.get('epoch', '?')} step={record.get('global_step', '?')}")
        return record

    def session_observer(record):
        global last_checkpoint_step, best_val_loss
        event_type = record.get("event_type", "")
        event = record.get("payload", {})

        if event_type == "stop_requested":
            return

        if event_type == "batch_end":
            batch_num = int(event.get("batch", 0))
            if batch_num > 0 and (batch_num % STDOUT_BATCH_INTERVAL == 0):
                print(
                    f"[TRAIN] epoch={event.get('epoch', 0)} "
                    f"batch={batch_num}/{event.get('total_batches', 0)} "
                    f"loss={event.get('batch_loss', 0.0):.4f} "
                    f"lr={event.get('lr', 0.0):.6f} "
                    f"speed={event.get('samples_per_sec', 0.0):.1f}s/s "
                    f"elapsed={event.get('elapsed_sec', 0.0):.0f}s eta={event.get('eta_sec', 0.0):.0f}s"
                )
            step = int(event.get("global_step", 0))
            if (
                CHECKPOINT_EVERY_N_STEPS > 0
                and step > 0
                and (step % CHECKPOINT_EVERY_N_STEPS == 0)
                and step != last_checkpoint_step
            ):
                _checkpoint(f"batch_{CHECKPOINT_EVERY_N_STEPS}", event)
                last_checkpoint_step = step

        if event_type == "epoch_end":
            train_loss_curve.append(float(event.get("epoch_loss", 0.0)))
            for key, curve in [
                ("val_loss", val_loss_curve),
                ("val_accuracy", val_acc_curve),
                ("macro_f1", macro_f1_curve),
                ("weighted_f1", weighted_f1_curve),
                ("balanced_accuracy", balanced_acc_curve),
                ("generalization_gap", gap_curve),
            ]:
                if key in event:
                    curve.append(float(event[key]))

            val_loss = float(event["val_loss"]) if "val_loss" in event else None
            mark_best = False
            if val_loss is not None and (best_val_loss is None or val_loss < best_val_loss):
                best_val_loss = val_loss
                STATE["best_val_loss"] = best_val_loss
                mark_best = True

            should_persist_curve = (
                mark_best
                or int(event["epoch_done"]) == 1
                or int(event["epoch_done"]) == int(EPOCHS)
                or bool(event.get("stopped_early", False))
                or (int(event["epoch_done"]) % 5 == 0)
            )
            if should_persist_curve:
                plt.figure(figsize=(13, 3))
                plt.subplot(1, 3, 1)
                plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o", label="Train")
                if val_loss_curve:
                    plt.plot(range(1, len(val_loss_curve) + 1), val_loss_curve, marker="s", label="Val")
                plt.xlabel("Epoch")
                plt.ylabel("Loss")
                plt.title("Loss")
                plt.grid(True, alpha=0.3)
                plt.legend()

                plt.subplot(1, 3, 2)
                for values, label, marker in [
                    (val_acc_curve, "Acc", "^"),
                    (macro_f1_curve, "MacroF1", "d"),
                    (weighted_f1_curve, "WtdF1", "x"),
                    (balanced_acc_curve, "BalAcc", "*"),
                ]:
                    if values:
                        plt.plot(range(1, len(values) + 1), values, marker=marker, label=label)
                plt.ylim(0, 1)
                plt.xlabel("Epoch")
                plt.ylabel("Score")
                plt.title("Metrics")
                plt.grid(True, alpha=0.3)
                plt.legend()

                plt.subplot(1, 3, 3)
                if gap_curve:
                    plt.plot(range(1, len(gap_curve) + 1), gap_curve, marker="o", label="Gap")
                plt.axhline(0, color="black", lw=1, alpha=0.5)
                plt.xlabel("Epoch")
                plt.ylabel("Gap")
                plt.title("Gen. Gap")
                plt.grid(True, alpha=0.3)
                plt.legend()
                plt.tight_layout()
                persist_training_curve_figure(
                    root=ROOT,
                    epoch_done=int(event["epoch_done"]),
                    telemetry=telemetry,
                )
                plt.close("all")

            _persist_history()

            if mark_best or bool(event.get("stopped_early", False)) or int(event["epoch_done"]) == int(EPOCHS) or CHECKPOINT_EVERY_N_STEPS <= 0:
                _checkpoint("epoch_end", event, mark_best=mark_best, val_loss=val_loss)

            parts = [f"[EPOCH] {event['epoch_done']}/{EPOCHS}: train_loss={event.get('epoch_loss', 0.0):.4f}"]
            if "val_loss" in event:
                parts.append(f"val_loss={event['val_loss']:.4f}")
            if "val_accuracy" in event:
                parts.append(f"val_acc={event['val_accuracy']:.4f}")
            if "macro_f1" in event:
                parts.append(f"macro_f1={event['macro_f1']:.4f}")
            if mark_best:
                parts.append("* BEST")
            print(" ".join(parts))
            telemetry.update_latest(
                {
                    "phase": "training",
                    "epoch_done": int(event["epoch_done"]),
                    "global_step": int(event.get("global_step", 0)),
                    "best_val_loss": best_val_loss,
                }
            )

    session = adapter.build_training_session(
        loaders["train"],
        num_epochs=EPOCHS,
        val_loader=val_loader,
        observers=[session_observer],
        resume_state=resume_state,
        run_id=run_id,
        validation_every_n_epochs=VALIDATION_EVERY_N_EPOCHS,
    )

    try:
        history = session.run()
        adapter.is_trained = True
    except Exception as exc:
        print(f"[TRAIN] Exception: {exc}")
        telemetry.emit_log(f"Training exception: {exc}", phase="train", level="error")
        if CHECKPOINT_ON_EXCEPTION:
            try:
                _checkpoint(
                    "exception",
                    {
                        "epoch": 0,
                        "batch": 0,
                        "global_step": int((STATE.get("history") or {}).get("global_step", 0)),
                        "elapsed_sec": time.time() - start_time,
                    },
                )
            except Exception:
                pass
        raise

    elapsed_total = time.time() - start_time
    STATE["history"] = history.to_dict()
    STATE["resume_state"] = session.snapshot_state()
    _persist_history()
    telemetry.update_latest(
        {
            "phase": "training_complete",
            "elapsed_sec": round(elapsed_total, 3),
            "stopped_early": bool(STATE["history"].get("stopped_early", False)),
        }
    )
    print(f"[TRAIN] Complete. elapsed={elapsed_total:.1f}s stopped_early={STATE['history'].get('stopped_early', False)}")

[TRAIN] epochs=30 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/81 loss=2.2574 lr=0.000300 speed=277.2s/s elapsed=149s eta=29995s
[TRAIN] epoch=1 batch=24/81 loss=2.0645 lr=0.000300 speed=277.4s/s elapsed=284s eta=28468s
[TRAIN] epoch=1 batch=36/81 loss=1.5915 lr=0.000300 speed=277.3s/s elapsed=426s eta=28319s
[TRAIN] epoch=1 batch=48/81 loss=1.2242 lr=0.000300 speed=277.3s/s elapsed=563s eta=27962s
[TRAIN] epoch=1 batch=60/81 loss=0.6758 lr=0.000300 speed=277.2s/s elapsed=696s eta=27482s
[TRAIN] epoch=1 batch=72/81 loss=0.6171 lr=0.000299 speed=276.9s/s elapsed=835s eta=27335s
[CKPT] epoch_end epoch=1 step=81
[EPOCH] 1/30: train_loss=1.4384 val_loss=0.5219 val_acc=0.8470 macro_f1=0.7561 * BEST
[TRAIN] epoch=2 batch=12/81 loss=0.4448 lr=0.000299 speed=275.2s/s elapsed=1167s eta=29313s
[TRAIN] epoch=2 batch=24/81 loss=0.4403 lr=0.000299 speed=275.2s/s elapsed=1176s eta=26037s
[TRAIN] epoch=2 batch=36/81 loss=0.4495 lr=0.000298 speed=275.7s/s elapsed=1183s eta=23387s
[TRAIN] epo

In [8]:
with TELEMETRY.capture_cell_output("Cell 7: OOD Calibration"):
    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    val_loader = STATE["loaders"].get("val")
    if val_loader is None or len(val_loader.dataset) == 0:
        raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

    calibration = adapter.calibrate_ood(val_loader)
    STATE["calibration"] = calibration

    num_classes = calibration.get("ood_calibration", {}).get("num_classes", 0)
    version = calibration.get("ood_calibration", {}).get("version", 0)
    print(f"[OOD] Calibration done. classes={num_classes} version={version}")
    TELEMETRY.update_latest(
        {
            "phase": "ood_calibrated",
            "ood_num_classes": num_classes,
            "ood_version": version,
        }
    )

[OOD] Calibration done. classes=12 version=1


In [9]:
with TELEMETRY.capture_cell_output("Cell 8: Adapter Save"):
    rt("Cell 8: adapter save started", phase="export")

    if STATE.get("adapter") is None:
        raise RuntimeError("Run Cell 5 first.")

    checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    STATE["adapter"].save_adapter(str(checkpoint_dir))
    asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

    print("Saved adapter directory:", asset_dir)
    if not asset_dir.exists():
        raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

    drive_adapter_root = TELEMETRY.artifacts_dir / "adapter_export" / "continual_sd_lora_adapter"
    for path in sorted(asset_dir.rglob("*")):
        if path.is_file():
            relative_path = path.relative_to(checkpoint_dir).as_posix()
            TELEMETRY.copy_artifact_file(path, f"adapter_export/{relative_path}")
            print(" -", path.relative_to(ROOT))

    print("Drive adapter directory:", drive_adapter_root)
    TELEMETRY.update_latest(
        {
            "phase": "adapter_saved",
            "adapter_dir": str(drive_adapter_root),
        }
    )
    rt("Cell 8: adapter save completed", phase="export")

Cell 8: adapter save started
Saved adapter directory: /content/bitirmeprojesi/outputs/colab_notebook_training/continual_sd_lora_adapter
 - outputs/colab_notebook_training/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/continual_sd_lora_adapter/fusion.pth
Drive adapter directory: /content/drive/MyDrive/aads_ulora/telemetry/20260317_075215_994111/artifacts/adapter_export/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
with TELEMETRY.capture_cell_output("Cell 9: Final Evaluation"):
    import json
    from scripts.colab_notebook_helpers import (
        build_notebook_completion_report,
        maybe_auto_disconnect_colab_runtime,
        notebook_artifact_root,
        persist_production_readiness_artifact,
        persist_validation_artifacts,
    )
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.training.services.ood_benchmark import run_leave_one_class_out_benchmark
    from src.training.validation import evaluate_model_with_artifact_metrics

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Run engine init cell first.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    test_loader = loaders.get("test")
    if test_loader is None or len(test_loader.dataset) == 0:
        raise RuntimeError("Test loader is empty. Final evaluation must use the held-out test split.")

    trainer = adapter._trainer
    if trainer is None:
        raise RuntimeError("Trainer not initialized.")

    notebook_config = json.loads(json.dumps(BASE_CONFIG))
    notebook_config.setdefault("training", {})["continual"] = json.loads(json.dumps(STATE["continual_config"]))
    evaluation_cfg = notebook_config.get("training", {}).get("continual", {}).get("evaluation", {})
    artifact_root = notebook_artifact_root(ROOT)

    trainer.adapter_model.eval()
    trainer.classifier.eval()
    trainer.fusion.eval()

    classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda item: item[1])]
    ood_loader = loaders.get("ood")

    def _evaluate_split(split_name: str, loader, *, artifact_subdir: str, label: str):
        evaluation = evaluate_model_with_artifact_metrics(trainer, loader, ood_loader=ood_loader)
        if evaluation is None:
            raise RuntimeError("No evaluation samples.")
        artifacts = persist_validation_artifacts(
            root=ROOT,
            y_true=evaluation.y_true,
            y_pred=evaluation.y_pred,
            classes=classes,
            telemetry=TELEMETRY,
            artifact_subdir=artifact_subdir,
            ood_labels=evaluation.ood_labels,
            ood_scores=evaluation.ood_scores,
            sure_ds_f1=evaluation.sure_ds_f1,
            conformal_empirical_coverage=evaluation.conformal_empirical_coverage,
            conformal_avg_set_size=evaluation.conformal_avg_set_size,
            context={"crop": CROP_NAME, "run_id": RUN_ID, "split_name": split_name, **evaluation.context},
        )
        metrics = artifacts["metric_gate"]["metrics"]
        extras = []
        if metrics.get("ood_auroc") is not None:
            extras.append(f"ood_auroc={float(metrics['ood_auroc']):.4f}")
        if metrics.get("sure_ds_f1") is not None:
            extras.append(f"sure_ds_f1={float(metrics['sure_ds_f1']):.4f}")
        if metrics.get("conformal_empirical_coverage") is not None:
            extras.append(f"conformal_cov={float(metrics['conformal_empirical_coverage']):.4f}")
        suffix = " " + " ".join(extras) if extras else ""
        accuracy = float(artifacts["report_dict"].get("accuracy", 0.0))
        print(f"[{label}] samples={len(evaluation.y_true)} classes={len(classes)} accuracy={accuracy:.4f}{suffix}")
        return artifacts

    results = {}
    val_loader = loaders.get("val")
    if val_loader is not None and len(val_loader.dataset) > 0:
        results["validation"] = _evaluate_split(
            "val",
            val_loader,
            artifact_subdir="validation",
            label="VALIDATION (reference)",
        )

    results["test"] = _evaluate_split(
        "test",
        test_loader,
        artifact_subdir="test",
        label="TEST (held-out)",
    )
    selected_split = "test" if "test" in results else "validation"
    selected_artifacts = results[selected_split]
    real_ood_present = ood_loader is not None and len(ood_loader.dataset) > 0
    ood_evidence_source = "real_ood_split" if real_ood_present else "unavailable"
    ood_evidence_metrics = dict(selected_artifacts["metric_gate"]["metrics"]) if real_ood_present else {}
    benchmark_summary = {}
    if (
        not real_ood_present
        and str(evaluation_cfg.get("ood_fallback_strategy", "held_out_benchmark")) == "held_out_benchmark"
        and bool(evaluation_cfg.get("ood_benchmark_auto_run", True))
    ):
        print("[OOD] No real OOD split detected; running held-out benchmark fallback...")
        benchmark_summary = run_leave_one_class_out_benchmark(
            crop_name=CROP_NAME,
            class_names=classes,
            loaders=loaders,
            config=notebook_config,
            device=DEVICE,
            artifact_root=artifact_root,
            adapter_factory=IndependentCropAdapter,
            run_id=RUN_ID,
            num_epochs=EPOCHS,
            telemetry=TELEMETRY,
            emit_event=lambda event_type, payload: TELEMETRY.emit_event(event_type, payload, phase="evaluation"),
            min_classes=int(evaluation_cfg.get("ood_benchmark_min_classes", 3)),
        )
        ood_evidence_source = "held_out_benchmark"
        ood_evidence_metrics = dict(benchmark_summary.get("metrics", {}))

    readiness = persist_production_readiness_artifact(
        root=ROOT,
        classification_metric_gate=selected_artifacts.get("metric_gate"),
        classification_split=selected_split,
        ood_evidence_source=ood_evidence_source,
        ood_metrics=ood_evidence_metrics,
        context={
            "run_id": RUN_ID,
            "crop_name": CROP_NAME,
            "classification_split": selected_split,
            "ood_benchmark_status": benchmark_summary.get("status"),
            "ood_benchmark_passed": benchmark_summary.get("passed"),
        },
        telemetry=TELEMETRY,
    )

    STATE["evaluation_artifacts"] = results
    STATE["ood_benchmark"] = benchmark_summary
    STATE["production_readiness"] = readiness["payload"]
    plt.close("all")
    print(
        f"[OOD] evidence={readiness['payload'].get('ood_evidence_source', 'unavailable')} "
        f"status={readiness['payload'].get('status', 'failed')} passed={bool(readiness['payload'].get('passed', False))}"
    )

    TELEMETRY.update_latest(
        {
            "phase": "evaluation_complete",
            "evaluation_splits": sorted(results.keys()),
            "ood_evidence_source": readiness["payload"].get("ood_evidence_source"),
            "production_readiness": readiness["payload"].get("status"),
        }
    )
    print("[DONE] Validation reference and held-out test artifacts saved to Drive.")

TELEMETRY.close(
    {
        "status": "ok",
        "evaluation_splits": sorted((STATE.get("evaluation_artifacts") or {}).keys()),
        "cell_outputs_dir": str(TELEMETRY.artifacts_dir / "cell_outputs"),
        "repo_run_dir": str(REPO_RUN_DIR),
    }
)

REPO_RUN_EXPORTS = save_run_outputs_to_repo()
for key in sorted(REPO_RUN_EXPORTS):
    print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")

print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
notebook_export_result = export_current_colab_notebook(REPO_NOTEBOOK_OUTPUT_PATH)
if AUTO_PUSH_TO_GITHUB:
    git_push_report = push_repo_run_to_github(
        ROOT,
        RUN_ID,
        remote_name=AUTO_PUSH_REMOTE_NAME,
        branch=AUTO_PUSH_BRANCH,
        print_fn=print,
    )
else:
    git_push_report = {"enabled": False, "pushed": False, "run_dir": str(REPO_RUN_DIR)}
STATE["git_push_report"] = git_push_report
disconnect_report = build_notebook_completion_report(
    state=STATE,
    telemetry=TELEMETRY,
    repo_run_exports=REPO_RUN_EXPORTS,
    notebook_export_path=notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH,
)
STATE["auto_disconnect_report"] = disconnect_report
print(f"[COLAB] completion checks -> {disconnect_report['checks']}")
maybe_auto_disconnect_colab_runtime(
    enabled=AUTO_DISCONNECT_RUNTIME,
    grace_period_sec=AUTO_DISCONNECT_GRACE_SECONDS,
    telemetry=TELEMETRY,
    completion_report=disconnect_report,
)

[VALIDATION (reference)] samples=1627 classes=12 accuracy=0.9773 ood_auroc=0.9848 sure_ds_f1=0.4989 conformal_cov=0.9803
[TEST (held-out)] samples=1642 classes=12 accuracy=0.9732 ood_auroc=0.9874 sure_ds_f1=0.5099 conformal_cov=0.9744
[OOD] evidence=real_ood_split status=failed passed=False
[DONE] Validation reference and held-out test artifacts saved to Drive.
